# Assignment 2 – Task 2: Heart Disease Prediction System
---
**Deployment Link:** [Heart Disease Predictor on Modal](https://mhassan0000--heart-disease-predictor-run.modal.run)

This notebook follows the **exact same steps** as Task 1 (Titanic Prediction):
1. Data Collection
2. Data Understanding (EDA)
3. Data Pre-processing & Label Encoding
4. Model Training (SVC)
5. Model Evaluation
6. Making Predictions
7. Saving the Trained Model

**Dataset:** UCI Heart Disease (Cleveland) – `heart.csv`
**Algorithm:** Support Vector Classifier (SVC)
**Constraint:** Only 4 input attributes selected from the dataset

### Selected 4 Input Attributes:
| Feature | Description |
|---------|-------------|
| `age` | Age of the patient (years) |
| `thalach` | Maximum heart rate achieved |
| `chol` | Serum cholesterol (mg/dl) |
| `cp` | Chest pain type (0-3) |


## Step 0 – Import Libraries

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")
print("Libraries imported successfully")

## Step 1 – Data Collection
Load the Heart Disease dataset from the local CSV file.

In [ ]:
df = pd.read_csv("heart.csv")
print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

## Step 2 – Data Understanding (EDA)

In [ ]:
print("=== Dataset Info ===")
df.info()

In [ ]:
print("=== Descriptive Statistics ===")
df.describe()

In [ ]:
print("=== Missing Values ===")
df.isnull().sum()

In [ ]:
print("=== Target Distribution (0=No Disease, 1=Disease) ===")
print(df["target"].value_counts())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Heart Disease Dataset – Exploratory Data Analysis", fontsize=14, fontweight="bold")

sns.countplot(data=df, x="target", ax=axes[0,0], palette="Set2")
axes[0,0].set_title("Heart Disease Count (0=No, 1=Yes)")

df[df["target"]==0]["age"].hist(bins=20, ax=axes[0,1], alpha=0.7, color="green", label="No Disease")
df[df["target"]==1]["age"].hist(bins=20, ax=axes[0,1], alpha=0.7, color="red", label="Disease")
axes[0,1].set_title("Age Distribution by Target")
axes[0,1].legend()

sns.boxplot(data=df, x="target", y="thalach", ax=axes[1,0], palette="Set1")
axes[1,0].set_title("Max Heart Rate by Target")
axes[1,0].set_xlabel("Target (0=No, 1=Yes)")

sns.countplot(data=df, x="cp", hue="target", ax=axes[1,1], palette="Set3")
axes[1,1].set_title("Chest Pain Type by Target")

plt.tight_layout()
plt.savefig("heart_disease_eda.png", dpi=100)
plt.show()
print("EDA plots saved")

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Heart Disease – Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("heart_disease_correlation.png", dpi=100)
plt.show()
print("Correlation heatmap saved")

## Step 3 – Data Pre-processing & Label Encoding

### Constraint: Select only 4 input attributes
Selected features: **age**, **thalach**, **chol**, **cp**

In [ ]:
# === SELECTED 4 FEATURES (Assignment Constraint) ===
SELECTED_FEATURES = ["age", "thalach", "chol", "cp"]
TARGET = "target"

print(f"Selected features: {SELECTED_FEATURES}")
print(f"Target: {TARGET}\n")

df_model = df[SELECTED_FEATURES + [TARGET]].copy()

# Handle missing values
print(f"Missing values:\n{df_model.isnull().sum()}")
df_model.dropna(inplace=True)
print(f"\nShape after handling nulls: {df_model.shape}")

# Label Encoding for chest pain type (cp)
le_cp = LabelEncoder()
df_model["cp"] = le_cp.fit_transform(df_model["cp"])
print(f"\nLabel Encoder classes for cp: {le_cp.classes_}")

# Save encoded dataset
df_model.to_csv("heart_disease_encoded.csv", index=False)
print("Encoded dataset saved to heart_disease_encoded.csv")
df_model.head()

## Step 4 – Model Training (Support Vector Classifier)

In [ ]:
X = df_model[SELECTED_FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")

svc_heart = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svc_heart.fit(X_train, y_train)
print("\nSVC model trained successfully")

## Step 5 – Model Evaluation

In [ ]:
y_pred = svc_heart.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
cm     = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc*100:.2f}%")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Disease","Disease"]))

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["No Disease","Disease"],
            yticklabels=["No Disease","Disease"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Heart Disease – Confusion Matrix")
plt.tight_layout()
plt.savefig("heart_disease_confusion_matrix.png", dpi=100)
plt.show()

## Step 6 – Making Predictions on Sample Data

In [ ]:
sample_data = pd.DataFrame({
    "age":     [52,   67,   45,   38,   60],
    "thalach": [168,  108,  150,  170,  120],
    "chol":    [212,  300,  225,  174,  280],
    "cp":      [0,    1,    2,    3,    0],
})
sample_encoded = sample_data.copy()
sample_encoded["cp"] = le_cp.transform(sample_encoded["cp"])

predictions = svc_heart.predict(sample_encoded)
sample_data["Predicted"] = ["Heart Disease" if p==1 else "No Heart Disease"
                              for p in predictions]

# Save predictions
sample_data.to_csv("heart_disease_predictions.csv", index=False)
print("Predictions saved to heart_disease_predictions.csv")
sample_data

## Step 7 – Save Trained Model

In [ ]:
# Save the SVC model
with open("heart_disease_svc_model.pkl", "wb") as f:
    pickle.dump(svc_heart, f)

# Save the LabelEncoder for 'cp'
with open("heart_disease_le_cp.pkl", "wb") as f:
    pickle.dump(le_cp, f)

print("Model saved to heart_disease_svc_model.pkl")
print("Label encoder saved to heart_disease_le_cp.pkl")
print("\n" + "="*50)
print("TASK 2 COMPLETE")
print("="*50)